# Factoid Standard DPO Training (Qwen 2.5 0.5B Full Single Answer)

This mirrors the existing Llama 8B standard pairwise-DPO experiment while starting from the Qwen 2.5 0.5B single-answer SFT adapter. It reuses the same chosen/rejected pairs and validation slates, but re-renders the prompts with Qwen chat tokens before reference scoring and training.

- Source data: one gold answer and existing wrong answers per question
- Training expansion: one pairwise record per chosen/negative pairing
- Validation: recompose each question into its fixed candidate slate for loss, generated BioASQ MRR, and top-1


In [ ]:
from __future__ import annotations



import gc

import os

import sys

import json

import random

import time

from pathlib import Path



os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

os.environ.setdefault("HF_HUB_OFFLINE", "1")

os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

os.environ.setdefault("HF_DATASETS_OFFLINE", "1")



import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

import torch

import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training

from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup, set_seed





def find_project_root(start: Path) -> Path:

    for candidate in [start, *start.parents]:

        if (candidate / "cse_dpo").exists() and (candidate / "src").exists():

            return candidate

    raise RuntimeError("Could not locate project root. Start the notebook inside the repo.")



PROJECT_ROOT = find_project_root(Path.cwd()).resolve()

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type != "cuda":

    raise RuntimeError("Please run this notebook on a CUDA GPU runtime.")

from cse_dpo.generated_bioasq_eval import build_generated_eval_rows, evaluate_generated_bioasq, load_gold_examples
from src.utility.bioasq_format import exact_answer_groups, normalize_for_bioasq_exact_match, parse_prediction_items
from src.prompt_registry import resolve_prompt_bundle
from src.utility.config import QUESTION_INSTRUCTIONS
from src.utility.eval_dataset import load_eval_examples, render_prompt

print("Using device:", DEVICE)

PROJECT_ROOT


In [ ]:
# Choose "existing_pairs" to reproduce the original experiment, or
# "evidence_strict_local" to use only the 666/89 question slates with four
# within-question generated negatives and no global backfill. Choose
# "evidence_all_supported" to use every supported question with one or more
# local negatives (currently 1,079/127); it never uses global backfill.
# "evidence_span_balanced_review" uses synthetic shorter/longer literal spans.
# "candidate_bank_gold_snippet_spans_review" preserves the original full-resource inputs.
# "candidate_bank_gold_snippet_spans_compact_review" uses only the individual
# gold-bearing [BS]...[ES] snippets that justify the selected sampled negatives.
# "candidate_bank_extractive_c3_standard" uses all extractive-C3 staged pairs converted to slates.
# "candidate_bank_stage_{discovery,discrimination,exactness}_standard" tests one stage at a time.
# Both span datasets require an explicit exploratory-run acknowledgement.
DATASET_VARIANT = "candidate_bank_stage_exactness_standard" 
ALLOW_UNREVIEWED_SPAN_CANDIDATES = True
ALLOW_UNREVIEWED_CANDIDATE_BANK_SPAN_CANDIDATES = False
# "match_dataset" preserves the corresponding SFT starting point. Select either
# explicit adapter to run a controlled data-vs-initialization comparison.
MODEL_VARIANT = "evidence_per_alias_sft_dropout_strict_extractive"  # Also: "evidence_per_alias_sft_dropout_strict_extractive".
# "match_model", "aligned", "strict_extractive", or "literal_copy".
PROMPT_VARIANT = "match_model"
# Use None for an automatically unique timestamp, or set a descriptive immutable label.
RUN_NAME = "candidate_bank_stage_exactness_standard_seed3407_run1_dropout_005_wd_001"  # Example: "evidence_softmax_seed3407_run1"

EXISTING_TRAIN_JSONL = PROJECT_ROOT / "Artifacts/cse_dpo/softmax_dpo_paper_style_llama8b/data_same_question_only_any_count/train_softmax_dpo.jsonl"
EXISTING_VALIDATION_JSONL = PROJECT_ROOT / "Artifacts/cse_dpo/softmax_dpo_paper_style_llama8b/data_dev_full_single_answer/validation_softmax_dpo.jsonl"
EXISTING_TRAIN_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources/train_prepared.json"
EXISTING_VALIDATION_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources/eval_prepared.json"
EXISTING_MODEL_REF = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/full_resources_single_answer_qwen25_05b/adapter"

EVIDENCE_PAIR_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/evidence_grounded_gold_response_pairs_qwen25_05b/run1"
EVIDENCE_TRAIN_JSONL = EVIDENCE_PAIR_ROOT / "train/question_classes.jsonl"
EVIDENCE_VALIDATION_JSONL = EVIDENCE_PAIR_ROOT / "dev/question_classes.jsonl"
EVIDENCE_ALL_SUPPORTED_PAIR_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/evidence_grounded_all_supported_pairs_qwen25_05b/run1"
EVIDENCE_ALL_SUPPORTED_TRAIN_JSONL = EVIDENCE_ALL_SUPPORTED_PAIR_ROOT / "train/question_classes.jsonl"
EVIDENCE_ALL_SUPPORTED_VALIDATION_JSONL = EVIDENCE_ALL_SUPPORTED_PAIR_ROOT / "dev/question_classes.jsonl"
SPAN_PAIR_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/span_dpo_pairs_qwen25_05b"
SPAN_TRAIN_JSONL = SPAN_PAIR_ROOT / "evidence_grounded_train_literal_span_balanced_v1/question_slates_needing_review.jsonl"
SPAN_VALIDATION_JSONL = SPAN_PAIR_ROOT / "evidence_grounded_dev_literal_span_balanced_v1/question_slates_needing_review.jsonl"
CANDIDATE_BANK_SPAN_TRAIN_JSONL = PROJECT_ROOT / "Artifacts/cse_dpo/candidate_banks/factoid_full_resources_single_answer_evidence_grounded_qwen25_05b_train_s3_n20/evidence-grounded-single-answer-qwen25-05b/span_mismatch_gold_snippet_extraction_v2/dpo_question_slates_needing_review.jsonl"
CANDIDATE_BANK_SPAN_VALIDATION_JSONL = PROJECT_ROOT / "Artifacts/cse_dpo/candidate_banks/factoid_full_resources_single_answer_evidence_grounded_qwen25_05b_dev_s3_n20/evidence-grounded-single-answer-qwen25-05b/span_mismatch_gold_snippet_extraction_v2/dpo_question_slates_needing_review.jsonl"
CANDIDATE_BANK_SPAN_TRAIN_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/train_prepared.json"
CANDIDATE_BANK_SPAN_VALIDATION_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/eval_prepared.json"
CANDIDATE_BANK_COMPACT_TRAIN_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/candidate_bank_gold_snippet_dpo_qwen25_05b/train_prepared.json"
CANDIDATE_BANK_COMPACT_VALIDATION_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/candidate_bank_gold_snippet_dpo_qwen25_05b/dev_prepared.json"
EXTRACTIVE_C3_SLATE_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/standard_dpo_slates_from_extract_c3_pairs/train_first400_extract_c3_primary_pairs_seed3407"
EXTRACTIVE_C3_TRAIN_JSONL = EXTRACTIVE_C3_SLATE_ROOT / "train_question_slates.jsonl"
EXTRACTIVE_C3_VALIDATION_JSONL = EXTRACTIVE_C3_SLATE_ROOT / "validation_question_slates.jsonl"
EXTRACTIVE_C3_TRAIN_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/train_prepared.json"
EXTRACTIVE_C3_VALIDATION_SOURCE_INPUT = EXTRACTIVE_C3_TRAIN_SOURCE_INPUT
STAGE_STANDARD_SLATE_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/standard_dpo_slates_from_extract_c3_stage_pairs"
STAGE_STANDARD_SOURCE_INPUT = EXTRACTIVE_C3_TRAIN_SOURCE_INPUT
EVIDENCE_TRAIN_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/evidence_grounded_single_answer_qwen25_05b/train_prepared.json"
EVIDENCE_VALIDATION_SOURCE_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/evidence_grounded_single_answer_qwen25_05b/dev_supported_prepared.json"
EVIDENCE_MODEL_REF = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_single_answer_qwen25_05b/adapter"
EVIDENCE_PER_ALIAS_MODEL_REF = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005/adapter"
STRICT_EXTRACTIVE_MODEL_REF = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/adapter"
FULL_PER_ALIAS_MODEL_REF = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/full_resources_per_gold_alias_qwen25_05b_lora_dropout_005/adapter"
PROMPT_REGISTRY_PATH = PROJECT_ROOT / "prompts/factoid_single_answer_aligned.json"
# All generated MRR values remain comparable on the full 160-question dev split.
FULL_DEV_EVAL_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/eval_prepared.json"

if DATASET_VARIANT == "existing_pairs":
    TRAIN_JSONL = EXISTING_TRAIN_JSONL
    VALIDATION_JSONL = EXISTING_VALIDATION_JSONL
    TRAIN_SOURCE_INPUT = EXISTING_TRAIN_SOURCE_INPUT
    VALIDATION_SOURCE_INPUT = EXISTING_VALIDATION_SOURCE_INPUT
    GENERATED_EVAL_SOURCE_INPUT = EXISTING_VALIDATION_SOURCE_INPUT
    DATASET_MODEL_REF = EXISTING_MODEL_REF
    DATASET_LABEL = "existing_pairs"
elif DATASET_VARIANT == "evidence_strict_local":
    TRAIN_JSONL = EVIDENCE_TRAIN_JSONL
    VALIDATION_JSONL = EVIDENCE_VALIDATION_JSONL
    TRAIN_SOURCE_INPUT = EVIDENCE_TRAIN_SOURCE_INPUT
    VALIDATION_SOURCE_INPUT = EVIDENCE_VALIDATION_SOURCE_INPUT
    GENERATED_EVAL_SOURCE_INPUT = FULL_DEV_EVAL_INPUT
    DATASET_MODEL_REF = EVIDENCE_MODEL_REF
    DATASET_LABEL = "evidence_strict_local_666_train_89_val"
elif DATASET_VARIANT == "evidence_all_supported":
    TRAIN_JSONL = EVIDENCE_ALL_SUPPORTED_TRAIN_JSONL
    VALIDATION_JSONL = EVIDENCE_ALL_SUPPORTED_VALIDATION_JSONL
    TRAIN_SOURCE_INPUT = EVIDENCE_TRAIN_SOURCE_INPUT
    VALIDATION_SOURCE_INPUT = EVIDENCE_VALIDATION_SOURCE_INPUT
    GENERATED_EVAL_SOURCE_INPUT = FULL_DEV_EVAL_INPUT
    DATASET_MODEL_REF = EVIDENCE_MODEL_REF
    DATASET_LABEL = "evidence_all_supported_local_only_1079_train_127_val"
elif DATASET_VARIANT == "evidence_span_balanced_review":
    if not ALLOW_UNREVIEWED_SPAN_CANDIDATES:
        raise ValueError("Span candidates require manual approval. Set ALLOW_UNREVIEWED_SPAN_CANDIDATES=True only for an explicitly labeled exploratory run.")
    TRAIN_JSONL = SPAN_TRAIN_JSONL
    VALIDATION_JSONL = SPAN_VALIDATION_JSONL
    TRAIN_SOURCE_INPUT = EVIDENCE_TRAIN_SOURCE_INPUT
    VALIDATION_SOURCE_INPUT = EVIDENCE_VALIDATION_SOURCE_INPUT
    GENERATED_EVAL_SOURCE_INPUT = FULL_DEV_EVAL_INPUT
    DATASET_MODEL_REF = STRICT_EXTRACTIVE_MODEL_REF
    DATASET_LABEL = "evidence_span_balanced_review_556_train_66_val"
elif DATASET_VARIANT == "candidate_bank_gold_snippet_spans_review":
    if not ALLOW_UNREVIEWED_CANDIDATE_BANK_SPAN_CANDIDATES:
        raise ValueError("Candidate-bank span pairs require explicit approval. Set ALLOW_UNREVIEWED_CANDIDATE_BANK_SPAN_CANDIDATES=True only for an explicitly labeled exploratory run.")
    TRAIN_JSONL = CANDIDATE_BANK_SPAN_TRAIN_JSONL
    VALIDATION_JSONL = CANDIDATE_BANK_SPAN_VALIDATION_JSONL
    TRAIN_SOURCE_INPUT = CANDIDATE_BANK_SPAN_TRAIN_SOURCE_INPUT
    VALIDATION_SOURCE_INPUT = CANDIDATE_BANK_SPAN_VALIDATION_SOURCE_INPUT
    GENERATED_EVAL_SOURCE_INPUT = FULL_DEV_EVAL_INPUT
    DATASET_MODEL_REF = STRICT_EXTRACTIVE_MODEL_REF
    DATASET_LABEL = "candidate_bank_gold_snippet_spans_review_509_train_75_val_full_resources"
elif DATASET_VARIANT == "candidate_bank_gold_snippet_spans_compact_review":
    if not ALLOW_UNREVIEWED_CANDIDATE_BANK_SPAN_CANDIDATES:
        raise ValueError("Candidate-bank span pairs require explicit approval. Set ALLOW_UNREVIEWED_CANDIDATE_BANK_SPAN_CANDIDATES=True only for an explicitly labeled exploratory run.")
    TRAIN_JSONL = CANDIDATE_BANK_SPAN_TRAIN_JSONL
    VALIDATION_JSONL = CANDIDATE_BANK_SPAN_VALIDATION_JSONL
    TRAIN_SOURCE_INPUT = CANDIDATE_BANK_COMPACT_TRAIN_SOURCE_INPUT
    VALIDATION_SOURCE_INPUT = CANDIDATE_BANK_COMPACT_VALIDATION_SOURCE_INPUT
    GENERATED_EVAL_SOURCE_INPUT = FULL_DEV_EVAL_INPUT
    DATASET_MODEL_REF = STRICT_EXTRACTIVE_MODEL_REF
    DATASET_LABEL = "candidate_bank_gold_snippet_spans_compact_review_509_train_75_val"
elif DATASET_VARIANT == "candidate_bank_extractive_c3_standard":
    TRAIN_JSONL = EXTRACTIVE_C3_TRAIN_JSONL
    VALIDATION_JSONL = EXTRACTIVE_C3_VALIDATION_JSONL
    TRAIN_SOURCE_INPUT = EXTRACTIVE_C3_TRAIN_SOURCE_INPUT
    VALIDATION_SOURCE_INPUT = EXTRACTIVE_C3_VALIDATION_SOURCE_INPUT
    GENERATED_EVAL_SOURCE_INPUT = FULL_DEV_EVAL_INPUT
    DATASET_MODEL_REF = STRICT_EXTRACTIVE_MODEL_REF
    DATASET_LABEL = "candidate_bank_extractive_c3_standard_164_train_41_val"
elif DATASET_VARIANT in {"candidate_bank_stage_discovery_standard", "candidate_bank_stage_discrimination_standard", "candidate_bank_stage_exactness_standard"}:
    STAGE_NAME = DATASET_VARIANT.replace("candidate_bank_stage_", "").replace("_standard", "")
    STAGE_SLATE_ROOT = STAGE_STANDARD_SLATE_ROOT / STAGE_NAME
    TRAIN_JSONL = STAGE_SLATE_ROOT / "train_question_slates.jsonl"
    VALIDATION_JSONL = STAGE_SLATE_ROOT / "validation_question_slates.jsonl"
    TRAIN_SOURCE_INPUT = STAGE_STANDARD_SOURCE_INPUT
    VALIDATION_SOURCE_INPUT = STAGE_STANDARD_SOURCE_INPUT
    GENERATED_EVAL_SOURCE_INPUT = FULL_DEV_EVAL_INPUT
    DATASET_MODEL_REF = STRICT_EXTRACTIVE_MODEL_REF
    DATASET_LABEL = f"candidate_bank_stage_{STAGE_NAME}_standard"
else:
    raise ValueError(f"Unsupported DATASET_VARIANT: {DATASET_VARIANT}")

VALIDATION_GOLD_EVAL_INPUT = FULL_DEV_EVAL_INPUT if DATASET_VARIANT != "existing_pairs" else VALIDATION_SOURCE_INPUT

if MODEL_VARIANT == "match_dataset":
    MODEL_REF = DATASET_MODEL_REF
    MODEL_LABEL = (
        "full_sft" if DATASET_VARIANT == "existing_pairs"
        else "evidence_per_alias_sft_dropout_strict_extractive"
        if DATASET_VARIANT in {"evidence_span_balanced_review", "candidate_bank_gold_snippet_spans_review", "candidate_bank_gold_snippet_spans_compact_review", "candidate_bank_extractive_c3_standard", "candidate_bank_stage_discovery_standard", "candidate_bank_stage_discrimination_standard", "candidate_bank_stage_exactness_standard"}
        else "evidence_sft"
    )
elif MODEL_VARIANT == "full_sft":
    MODEL_REF = EXISTING_MODEL_REF
    MODEL_LABEL = "full_sft"
elif MODEL_VARIANT == "evidence_sft":
    MODEL_REF = EVIDENCE_MODEL_REF
    MODEL_LABEL = "evidence_sft"
elif MODEL_VARIANT == "evidence_per_alias_sft_dropout":
    MODEL_REF = EVIDENCE_PER_ALIAS_MODEL_REF
    MODEL_LABEL = "evidence_per_alias_sft_dropout"
elif MODEL_VARIANT == "evidence_per_alias_sft_dropout_strict_extractive":
    MODEL_REF = STRICT_EXTRACTIVE_MODEL_REF
    MODEL_LABEL = "evidence_per_alias_sft_dropout_strict_extractive"
elif MODEL_VARIANT == "full_per_alias_sft_dropout":
    MODEL_REF = FULL_PER_ALIAS_MODEL_REF
    MODEL_LABEL = "full_per_alias_sft_dropout"
else:
    raise ValueError(f"Unsupported MODEL_VARIANT: {MODEL_VARIANT}")

PROMPT_REF_BY_VARIANT = {
    "aligned": "factoid-single-answer-aligned-v1",
    "strict_extractive": "factoid-single-answer-extractive-v1",
    "literal_copy": "factoid-single-answer-literal-copy-v1",
}
if PROMPT_VARIANT == "match_model":
    DPO_PROMPT_REF = ("factoid-single-answer-extractive-v1" if MODEL_LABEL == "evidence_per_alias_sft_dropout_strict_extractive" else "factoid-single-answer-aligned-v1")
elif PROMPT_VARIANT in PROMPT_REF_BY_VARIANT:
    DPO_PROMPT_REF = PROMPT_REF_BY_VARIANT[PROMPT_VARIANT]
else:
    raise ValueError(f"Unsupported PROMPT_VARIANT: {PROMPT_VARIANT}")
DPO_PROMPT_BUNDLE = resolve_prompt_bundle(PROMPT_REGISTRY_PATH, DPO_PROMPT_REF, QUESTION_INSTRUCTIONS)
DPO_PROMPT_ID = DPO_PROMPT_BUNDLE["prompt_id"]
DPO_PROMPT_INSTRUCTIONS = DPO_PROMPT_BUNDLE["instructions"]

# BASE_MODEL_REF = (PROJECT_ROOT / "models/Qwen2.5-0.5B-Instruct").resolve()
STRICT_EXTRACTIVE_MODEL_REF = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_3b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr"
BASE_MODEL_REF = (PROJECT_ROOT / "models/Qwen2.5-3B-Instruct").resolve()

# New directory prevents reuse of reference scores cached during the 32k-context attempt.
BASE_OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_full_single_answer_existing_pairs_run1_ctx4096"
if DATASET_VARIANT == "evidence_strict_local":
    BASE_OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_evidence_strict_local_run1_ctx4096"
elif DATASET_VARIANT == "evidence_all_supported":
    BASE_OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_evidence_all_supported_local_only_run1_ctx4096"
elif DATASET_VARIANT == "evidence_span_balanced_review":
    BASE_OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_evidence_span_balanced_review_run1_ctx4096"
elif DATASET_VARIANT == "candidate_bank_gold_snippet_spans_review":
    BASE_OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_candidate_bank_gold_snippet_spans_review_run1_ctx4096"

elif DATASET_VARIANT == "candidate_bank_gold_snippet_spans_compact_review":
    BASE_OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_candidate_bank_gold_snippet_spans_compact_review_run1_ctx4096"
elif DATASET_VARIANT == "candidate_bank_extractive_c3_standard":
    BASE_OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_candidate_bank_extractive_c3_run1_ctx9000"
elif DATASET_VARIANT in {"candidate_bank_stage_discovery_standard", "candidate_bank_stage_discrimination_standard", "candidate_bank_stage_exactness_standard"}:
    stage_name_for_output = DATASET_VARIANT.replace("candidate_bank_stage_", "").replace("_standard", "")
    BASE_OUTPUT_DIR = PROJECT_ROOT / f"Artifacts/cse_dpo/pairwise_dpo_qwen25_05b/runs/standard_dpo_qwen25_05b_candidate_bank_stage_{stage_name_for_output}_run1_ctx9000"

# An explicit model override receives an isolated output directory so frozen
# reference scores from another adapter can never be reused.
if MODEL_VARIANT != "match_dataset":
    BASE_OUTPUT_DIR = BASE_OUTPUT_DIR.with_name(f"{BASE_OUTPUT_DIR.name}_model_{MODEL_LABEL}")

# Keep every invocation isolated. A named run is never overwritten accidentally.
if RUN_NAME is None:
    RUN_NAME = time.strftime("run_%Y%m%d_%H%M%S")
if not isinstance(RUN_NAME, str) or not RUN_NAME.strip() or any(char in RUN_NAME for char in "/\\"):
    raise ValueError("RUN_NAME must be None or a non-empty label without path separators")
BASE_OUTPUT_DIR = BASE_OUTPUT_DIR.with_name(f"{BASE_OUTPUT_DIR.name}_{RUN_NAME.strip()}")
print(f"Dataset: {DATASET_LABEL}; model: {MODEL_LABEL}; prompt: {DPO_PROMPT_ID}")

SMOKE_TEST = False
SMOKE_TRAIN_QUESTIONS = 8
SMOKE_VAL_QUESTIONS = 8
SMOKE_TAG = f"smoke_t{SMOKE_TRAIN_QUESTIONS}_v{SMOKE_VAL_QUESTIONS}"
OUTPUT_DIR = BASE_OUTPUT_DIR.with_name(f"{BASE_OUTPUT_DIR.name}_{SMOKE_TAG}") if SMOKE_TEST else BASE_OUTPUT_DIR

SEED = 3407
LEARNING_RATE = 1e-6
WEIGHT_DECAY = 0.01
NUM_TRAIN_EPOCHS = 8
BETA = 0.05
# Keep all resources, but cap model context at the SFT-proven 4,096 tokens.
# Full 32k-context DPO backpropagation exceeds the available 15.6 GB GPU memory.
MAX_SEQ_LENGTH = 9000
MAX_PROMPT_RESOURCES = 0
SCORING_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 64
EVAL_EVERY_STEPS = 1 if SMOKE_TEST else 25
WARMUP_STEPS = 5 if SMOKE_TEST else 150
EARLY_STOPPING_PATIENCE = 1 if SMOKE_TEST else 3
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
ATTN_IMPLEMENTATION = "sdpa"
LOCAL_FILES_ONLY = True
FORCE_REFERENCE_RECACHE = True if SMOKE_TEST else False
EMPTY_CACHE_EVERY_STEPS = 1
GENERATED_EVAL_MAX_NEW_TOKENS = 64
GENERATED_EVAL_DO_SAMPLE = False
GENERATED_EVAL_TEMPERATURE = 1.0
GENERATED_EVAL_TOP_P = 1.0
GENERATED_EVAL_SUMMARY_REFERENCE_MODE = "all-mean"
GENERATED_EVAL_EMPTY_CUDA_CACHE = False

if OUTPUT_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite an existing run directory: {OUTPUT_DIR}. Choose another RUN_NAME.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

for path in (TRAIN_JSONL, VALIDATION_JSONL, TRAIN_SOURCE_INPUT, VALIDATION_SOURCE_INPUT, GENERATED_EVAL_SOURCE_INPUT, VALIDATION_GOLD_EVAL_INPUT, MODEL_REF / "adapter_config.json", BASE_MODEL_REF / "config.json"):
    assert path.exists(), f"Missing required input: {path}"

print("DATASET_VARIANT:", DATASET_VARIANT)
print("DATASET_LABEL:", DATASET_LABEL)
print("MODEL_VARIANT:", MODEL_VARIANT)
print("MODEL_LABEL:", MODEL_LABEL)
print("MODEL_REF:", MODEL_REF)
print("BASE_MODEL_REF:", BASE_MODEL_REF)
print("RUN_NAME:", RUN_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SMOKE_TEST:", SMOKE_TEST)
print("LEARNING_RATE:", LEARNING_RATE)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("BETA:", BETA)
print("EFFECTIVE_BATCH_SIZE:", SCORING_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("OPTIMIZER: RMSprop")
print("WARMUP_STEPS:", WARMUP_STEPS)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("MAX_PROMPT_RESOURCES:", "all" if MAX_PROMPT_RESOURCES == 0 else MAX_PROMPT_RESOURCES)


In [ ]:
def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_question_slate_rows(rows):
    """Accept both legacy Softmax-DPO slates and strict-local factoid exports."""
    normalized_rows = []
    for row in rows:
        chosen = row.get("chosen") or row.get("chosen_output")
        negatives = row.get("negatives") or row.get("wrong_outputs")
        if not isinstance(chosen, str) or not chosen.strip():
            raise ValueError(f"Missing chosen completion for {row.get('question_id')}")
        if not isinstance(negatives, list) or not negatives:
            raise ValueError(f"Missing negative completions for {row.get('question_id')}")
        normalized_rows.append({**row, "chosen": chosen, "negatives": negatives})
    return normalized_rows

all_train_question_rows = normalize_question_slate_rows(load_jsonl(TRAIN_JSONL))
all_validation_question_rows = normalize_question_slate_rows(load_jsonl(VALIDATION_JSONL))
train_question_rows = all_train_question_rows[: min(SMOKE_TRAIN_QUESTIONS, len(all_train_question_rows))] if SMOKE_TEST else all_train_question_rows
validation_question_rows = all_validation_question_rows[: min(SMOKE_VAL_QUESTIONS, len(all_validation_question_rows))] if SMOKE_TEST else all_validation_question_rows
print(f"Loaded train questions: {len(all_train_question_rows):,}")
print(f"Loaded validation questions: {len(all_validation_question_rows):,}")
print(f"Active train questions: {len(train_question_rows):,}")
print(f"Active validation questions: {len(validation_question_rows):,}")
print(f"Expected train pairs: {sum(len(r['negatives']) for r in train_question_rows):,}")
if train_question_rows:
    print(json.dumps(train_question_rows[0], indent=2)[:1400])


In [ ]:
def resolve_local_model_ref(model_ref: Path | str, *, require_adapter: bool = False, require_config: bool = False):

    path = Path(model_ref)

    if not path.is_absolute():

        path = (PROJECT_ROOT / path).resolve()

    else:

        path = path.resolve()

    if not path.exists():

        raise FileNotFoundError(f"Local model path does not exist: {path}")

    if require_adapter and not (path / "adapter_config.json").exists():

        raise FileNotFoundError(f"Expected adapter_config.json under {path}")

    if require_config and not (path / "config.json").exists():

        raise FileNotFoundError(f"Expected config.json under {path}")

    return path





def load_tokenizer(model_ref: Path | str, base_model_ref: Path | str | None = None):

    model_path = resolve_local_model_ref(model_ref)

    if (model_path / "tokenizer.json").exists() or (model_path / "tokenizer_config.json").exists():

        load_path = model_path

    else:

        if base_model_ref is None:

            raise FileNotFoundError(f"No tokenizer files found under {model_path} and no base model path was provided.")

        load_path = resolve_local_model_ref(base_model_ref, require_config=True)

    tokenizer = AutoTokenizer.from_pretrained(str(load_path), local_files_only=LOCAL_FILES_ONLY)

    if tokenizer.pad_token_id is None:

        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    return tokenizer





def release_cuda_objects(*names: str):

    for name in names:

        if name in globals():

            try:

                del globals()[name]

            except Exception:

                pass

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()





def load_base_model(base_model_ref: Path | str):

    base_model_path = resolve_local_model_ref(base_model_ref, require_config=True)

    release_cuda_objects()

    if DEVICE.type == "cuda":

        quantization_config = BitsAndBytesConfig(

            load_in_4bit=True,

            bnb_4bit_quant_type="nf4",

            bnb_4bit_use_double_quant=True,

            bnb_4bit_compute_dtype=torch.float16,

        )

        model = AutoModelForCausalLM.from_pretrained(

            str(base_model_path),

            quantization_config=quantization_config,

            attn_implementation=ATTN_IMPLEMENTATION,

            device_map={"": DEVICE.index or 0},

            low_cpu_mem_usage=True,

            local_files_only=LOCAL_FILES_ONLY,

        )

    else:

        model = AutoModelForCausalLM.from_pretrained(

            str(base_model_path),

            torch_dtype=torch.float32,

            attn_implementation=ATTN_IMPLEMENTATION,

            low_cpu_mem_usage=True,

            local_files_only=LOCAL_FILES_ONLY,

        )

    model.config.use_cache = False

    return model





def load_reference_model(model_ref: Path | str, base_model_ref: Path | str):

    adapter_path = resolve_local_model_ref(model_ref, require_adapter=True)

    model = load_base_model(base_model_ref)

    model = PeftModel.from_pretrained(model, str(adapter_path), local_files_only=LOCAL_FILES_ONLY)

    model.eval()

    model.config.use_cache = False

    return model





def load_policy_model(model_ref: Path | str, base_model_ref: Path | str):

    adapter_path = resolve_local_model_ref(model_ref, require_adapter=True)

    model = load_base_model(base_model_ref)

    model = prepare_model_for_kbit_training(model)

    model = PeftModel.from_pretrained(

        model,

        str(adapter_path),

        local_files_only=LOCAL_FILES_ONLY,

        is_trainable=True,

    )

    model.config.use_cache = False

    if hasattr(model, "enable_input_require_grads"):

        model.enable_input_require_grads()

    model.gradient_checkpointing_enable()

    model.train()

    return model





release_cuda_objects("reference_model", "policy_model")

tokenizer = load_tokenizer(MODEL_REF, BASE_MODEL_REF)

def load_source_examples(path: Path):
    source_args = type("SourceArgs", (), {
        "question_types": ["factoid"],
        "max_resources": MAX_PROMPT_RESOURCES,
        "max_resource_chars": 0,
        "resource_selection": "first",
        "resource_granularity": "document",
        "resource_window_mode": "single",
        "resource_reranker_model": None,
        "resource_reranker_article_model": None,
        "resource_reranker_device": None,
        "resource_reranker_batch_size": 1,
        "local_files_only": LOCAL_FILES_ONLY,
        "limit": None,
    })()
    return {example.question_id: example for example in load_eval_examples([path], source_args, DPO_PROMPT_INSTRUCTIONS)}

def rerender_rows_for_qwen(rows, source_input: Path):
    examples_by_id = load_source_examples(source_input)
    rerendered = []
    missing_ids = []
    for row in rows:
        example = examples_by_id.get(row["question_id"])
        if example is None:
            missing_ids.append(row["question_id"])
            continue
        rerendered.append({
            **row,
            "prompt": render_prompt(tokenizer, example, chat_template="qwen-2.5", prompt_format="chat"),
        })
    if missing_ids:
        raise ValueError(f"Could not rebuild Qwen prompts for {len(missing_ids)} pair rows; first IDs: {missing_ids[:5]}")
    return rerendered

train_question_rows = rerender_rows_for_qwen(train_question_rows, TRAIN_SOURCE_INPUT)
validation_question_rows = rerender_rows_for_qwen(validation_question_rows, VALIDATION_SOURCE_INPUT)
validation_gold_examples = load_gold_examples([VALIDATION_GOLD_EVAL_INPUT])
# Generated BioASQ MRR uses all 160 dev questions; fixed-slate loss uses the selected DPO validation slates.
generated_eval_prompt_rows = [{"question_id": example.question_id, "prompt": render_prompt(tokenizer, example, chat_template="qwen-2.5", prompt_format="chat")} for example in load_source_examples(GENERATED_EVAL_SOURCE_INPUT).values()]
generated_validation_rows, missing_validation_gold_ids = build_generated_eval_rows(generated_eval_prompt_rows, validation_gold_examples)

def completion_aliases(completion):
    return {normalize_for_bioasq_exact_match(value) for value in parse_prediction_items(str(completion), "factoid") if normalize_for_bioasq_exact_match(value)}

def assert_dpo_data_invariants(train_rows, validation_rows):
    train_ids = [row["question_id"] for row in train_rows]
    validation_ids = [row["question_id"] for row in validation_rows]
    assert len(train_ids) == len(set(train_ids)), "Duplicate training question IDs"
    assert len(validation_ids) == len(set(validation_ids)), "Duplicate validation question IDs"
    assert not set(train_ids) & set(validation_ids), "Train/validation question ID overlap"
    for split_name, rows, source_input in [("train", train_rows, TRAIN_SOURCE_INPUT), ("validation", validation_rows, VALIDATION_SOURCE_INPUT)]:
        examples_by_id = load_source_examples(source_input)
        for row in rows:
            question_id = row["question_id"]
            example = examples_by_id.get(question_id)
            assert example is not None, f"{split_name}: missing source question {question_id}"
            prompt = row["prompt"]
            assert example.body in prompt, f"{split_name}: prompt lost question text for {question_id}"
            assert "<|im_start|>assistant\nAnswer:" in prompt and prompt.rstrip().endswith("Answer:"), f"{split_name}: missing Qwen assistant boundary for {question_id}"
            chosen_aliases = completion_aliases(row["chosen"])
            gold_aliases = {normalize_for_bioasq_exact_match(alias) for group in exact_answer_groups(example, "factoid") for alias in group}
            assert chosen_aliases, f"{split_name}: empty chosen completion for {question_id}"
            for negative_index, negative in enumerate(row["negatives"]):
                assert isinstance(negative, str) and negative.strip(), f"{split_name}: empty negative {negative_index} for {question_id}"
                negative_aliases = completion_aliases(negative)
                assert not (negative_aliases & chosen_aliases), f"{split_name}: chosen duplicated among negatives for {question_id}"
                assert not (negative_aliases & gold_aliases), f"{split_name}: gold alias appears as negative for {question_id}"

def assert_context_budget(rows, split_name):
    overflows = []
    for row in rows:
        prompt_length = len(tokenizer(row["prompt"], add_special_tokens=False).input_ids)
        for completion_name, completion in [("chosen", row["chosen"])] + [(f"negative_{index}", value) for index, value in enumerate(row["negatives"])]:
            completion_length = len(tokenizer(completion, add_special_tokens=False).input_ids) + int(tokenizer.eos_token_id is not None)
            available_prompt_length = MAX_SEQ_LENGTH - completion_length
            if prompt_length > available_prompt_length:
                overflows.append((row["question_id"], completion_name, prompt_length, completion_length, available_prompt_length))
    if overflows:
        preview = "; ".join(f"{qid}/{name}: prompt={prompt_tokens}, completion={completion_tokens}, available={available}" for qid, name, prompt_tokens, completion_tokens, available in overflows[:10])
        raise ValueError(f"{split_name}: {len(overflows)} prompt/completion pairs exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. Reduce resources before rendering. First examples: {preview}")

assert_dpo_data_invariants(train_question_rows, validation_question_rows)
assert_context_budget(train_question_rows, "train")
assert_context_budget(validation_question_rows, "validation")

print(f"Qwen-rendered train rows: {len(train_question_rows):,}")
print(f"Qwen-rendered validation rows: {len(validation_question_rows):,}")
print(f"Generated-eval dev questions: {len(generated_validation_rows):,}")
print(f"Generated-eval missing gold IDs: {len(missing_validation_gold_ids):,}")

reference_model = load_reference_model(MODEL_REF, BASE_MODEL_REF)

policy_model = None

print(type(reference_model).__name__)


In [ ]:
def encode_pair(prompt: str, completion: str):

    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids

    completion_ids = tokenizer(completion, add_special_tokens=False).input_ids

    if tokenizer.eos_token_id is not None:

        completion_ids = list(completion_ids) + [int(tokenizer.eos_token_id)]

    if len(completion_ids) >= MAX_SEQ_LENGTH:

        raise ValueError("Completion alone exceeds MAX_SEQ_LENGTH")

    available_prompt_length = MAX_SEQ_LENGTH - len(completion_ids)
    if len(prompt_ids) > available_prompt_length:
        raise ValueError(
            f"Prompt has {len(prompt_ids)} tokens but only {available_prompt_length} are available for this completion. "
            "Reduce resources before rendering; do not silently truncate the prompt."
        )

    input_ids = prompt_ids + completion_ids

    labels = [-100] * len(prompt_ids) + completion_ids

    return input_ids, labels





def collate_pairs(pairs):

    encoded = [encode_pair(prompt, completion) for prompt, completion in pairs]

    width = max(len(input_ids) for input_ids, _ in encoded)

    input_rows, mask_rows, label_rows = [], [], []

    for input_ids, labels in encoded:

        pad = width - len(input_ids)

        input_rows.append(input_ids + [int(tokenizer.pad_token_id)] * pad)

        mask_rows.append([1] * len(input_ids) + [0] * pad)

        label_rows.append(labels + [-100] * pad)

    return {

        "input_ids": torch.tensor(input_rows, dtype=torch.long, device=DEVICE),

        "attention_mask": torch.tensor(mask_rows, dtype=torch.long, device=DEVICE),

        "labels": torch.tensor(label_rows, dtype=torch.long, device=DEVICE),

    }





def sequence_logps(model, pairs, require_grad=False):

    outputs = []

    context = torch.enable_grad() if require_grad else torch.no_grad()

    with context:

        for start in range(0, len(pairs), SCORING_BATCH_SIZE):

            batch = collate_pairs(pairs[start:start + SCORING_BATCH_SIZE])

            if batch["input_ids"].size(0) != 1:

                raise ValueError("SCORING_BATCH_SIZE must remain 1 when using logits_to_keep")

            targets = batch["labels"]

            target_positions = torch.nonzero(targets[0] != -100, as_tuple=False).squeeze(-1)

            logit_positions = target_positions - 1

            # DPO scores only completion tokens; avoid materializing prompt-token logits.
            result = model(

                input_ids=batch["input_ids"],

                attention_mask=batch["attention_mask"],

                logits_to_keep=logit_positions,

                use_cache=False,

            )

            logits = result.logits

            targets = targets[:, target_positions].contiguous()

            seq_neg_logps = torch.zeros(logits.size(0), dtype=torch.float32, device=logits.device)

            for chunk_start in range(0, logits.size(1), 128):

                chunk_end = min(chunk_start + 128, logits.size(1))

                chunk_logits = logits[:, chunk_start:chunk_end, :]

                chunk_targets = targets[:, chunk_start:chunk_end]

                valid_targets = chunk_targets != -100

                safe_targets = chunk_targets.masked_fill(~valid_targets, 0)

                def chunk_token_logps(chunk_logits, safe_targets):

                    chunk_logits = chunk_logits.float()

                    selected_logits = torch.gather(

                        chunk_logits,

                        dim=-1,

                        index=safe_targets.unsqueeze(-1),

                    ).squeeze(-1)

                    return selected_logits - torch.logsumexp(chunk_logits, dim=-1)

                if require_grad:

                    # Recompute vocabulary-wide scoring intermediates in backward.
                    token_logps = checkpoint(chunk_token_logps, chunk_logits, safe_targets, use_reentrant=False)

                else:

                    token_logps = chunk_token_logps(chunk_logits, safe_targets)

                seq_neg_logps = seq_neg_logps - (token_logps * valid_targets).sum(dim=-1)

                del chunk_logits, chunk_targets, valid_targets, safe_targets, token_logps

            outputs.append((-seq_neg_logps).float())

            del batch, result, logits, targets, seq_neg_logps

    return torch.cat(outputs, dim=0)





def reference_cache_rows(rows, split: str):

    cache_rows = []

    for row in tqdm(rows, desc=f"Reference cache {split}"):

        pairs = [(row["prompt"], row["chosen"])] + [(row["prompt"], negative) for negative in row["negatives"]]

        scores = sequence_logps(reference_model, pairs, require_grad=False).detach().cpu().tolist()

        cache_rows.append({

            **row,

            "reference_chosen_logp_sum": float(scores[0]),

            "reference_negative_logp_sums": [float(x) for x in scores[1:]],

        })

    return cache_rows



def trim_prompt_to_first_n_resources(prompt: str, max_resources: int) -> str:

    if max_resources <= 0:

        raise ValueError("max_resources must be positive")

    lines = prompt.splitlines()

    trimmed_lines = []

    resource_count = 0

    in_resources = False

    for line in lines:

        if line.startswith("Resource "):

            in_resources = True

            resource_count += 1

            if resource_count > max_resources:

                break

        trimmed_lines.append(line)

    if not in_resources:

        return prompt

    return "\n".join(trimmed_lines)



def trim_cached_rows_to_first_n_resources(rows, max_resources: int):

    trimmed_rows = []

    for row in rows:

        trimmed_rows.append({

            **row,

            "prompt": trim_prompt_to_first_n_resources(row["prompt"], max_resources),

        })

    return trimmed_rows





def write_json(path: Path, payload):

    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as handle:

        json.dump(payload, handle, indent=2, ensure_ascii=False, sort_keys=True)





def write_jsonl(path: Path, rows):

    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as handle:

        for row in rows:

            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True))

            handle.write("\n")





train_cache_path = OUTPUT_DIR / "train_questions_with_reference.jsonl"

validation_cache_path = OUTPUT_DIR / "validation_questions_with_reference.jsonl"



if (

    not FORCE_REFERENCE_RECACHE

    and train_cache_path.exists()

    and validation_cache_path.exists()

):

    print("Loading existing reference cache from disk")

    train_cached = load_jsonl(train_cache_path)

    validation_cached = load_jsonl(validation_cache_path)

else:

    train_cached = reference_cache_rows(train_question_rows, "train")

    validation_cached = reference_cache_rows(validation_question_rows, "validation")

    write_jsonl(train_cache_path, train_cached)

    write_jsonl(validation_cache_path, validation_cached)

    print("Reference cache complete and saved to disk")





def assert_reference_cache_invariants(rows, split_name):
    for row in rows:
        assert len(row["reference_negative_logp_sums"]) == len(row["negatives"]), f"{split_name}: reference-score count mismatch for {row['question_id']}"
        float(row["reference_chosen_logp_sum"])
        for score in row["reference_negative_logp_sums"]:
            float(score)

assert_reference_cache_invariants(train_cached, "train")
assert_reference_cache_invariants(validation_cached, "validation")

pairwise_train_rows = []

for row in train_cached:

    for negative_index, negative in enumerate(row["negatives"]):

        pairwise_train_rows.append({

            "question_id": row["question_id"],

            "prompt": row["prompt"],

            "chosen": row["chosen"],

            "rejected": negative,

            "reference_chosen_logp_sum": row["reference_chosen_logp_sum"],

            "reference_rejected_logp_sum": row["reference_negative_logp_sums"][negative_index],

            "negative_index": negative_index,

        })



print(f"Pairwise train rows: {len(pairwise_train_rows):,}")



release_cuda_objects("reference_model")

reference_model = None

policy_model = load_policy_model(MODEL_REF, BASE_MODEL_REF)

print(type(policy_model).__name__)


In [ ]:
def pairwise_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected):

    chosen_reward = policy_chosen - ref_chosen

    rejected_reward = policy_rejected - ref_rejected

    loss = -F.logsigmoid(BETA * (chosen_reward - rejected_reward))

    return loss, chosen_reward, rejected_reward





def evaluate_fixed_slate(rows):

    policy_model.eval()

    metrics = []

    with torch.no_grad():

        for row in tqdm(rows, desc="Validation", leave=False):

            pairs = [(row["prompt"], row["chosen"])] + [(row["prompt"], negative) for negative in row["negatives"]]

            policy_scores = sequence_logps(policy_model, pairs, require_grad=False)

            ref_chosen = torch.tensor(row["reference_chosen_logp_sum"], dtype=torch.float32, device=DEVICE)

            ref_negs = torch.tensor(row["reference_negative_logp_sums"], dtype=torch.float32, device=DEVICE)

            rewards = policy_scores - torch.cat([ref_chosen.view(1), ref_negs], dim=0)

            pair_losses = []

            for idx in range(1, len(rewards)):

                pair_loss, *_ = pairwise_loss(policy_scores[0], policy_scores[idx], ref_chosen, ref_negs[idx - 1])

                pair_losses.append(pair_loss)

            loss = torch.stack(pair_losses).mean()

            policy_rank = int(torch.argsort(policy_scores, descending=True).tolist().index(0) + 1)

            reward_margins = rewards[0] - rewards[1:]
            pairwise_reward_accuracy = ((reward_margins > 1e-6).float() + 0.5 * (reward_margins.abs() <= 1e-6).float()).mean()
            mean_reward_margin = reward_margins.mean()

            policy_slate_probs = torch.softmax(policy_scores, dim=0)

            dpo_reward_probs = torch.softmax(BETA * rewards, dim=0)

            metrics.append({

                "question_id": row["question_id"],

                "loss": float(loss.cpu()),

                "policy_logp_sum_chosen_rank": policy_rank,

                "policy_logp_sum_reciprocal_rank": 1.0 / policy_rank,

                "policy_logp_sum_top1": float(policy_rank == 1),

                "policy_logp_sum_slate_probability": float(policy_slate_probs[0].cpu()),

                "dpo_reward_pairwise_accuracy": float(pairwise_reward_accuracy.cpu()),

                "dpo_reward_mean_margin": float(mean_reward_margin.cpu()),

                "dpo_reward_slate_probability": float(dpo_reward_probs[0].cpu()),

            })

            del pairs, policy_scores, ref_chosen, ref_negs, rewards, reward_margins, pair_losses, loss, policy_slate_probs, dpo_reward_probs, pairwise_reward_accuracy, mean_reward_margin

            if EMPTY_CACHE_EVERY_STEPS > 0:

                gc.collect()

                torch.cuda.empty_cache()

    policy_model.train()

    summary = {

        "question_count": len(metrics),

        "val_loss": float(np.mean([m["loss"] for m in metrics])) if metrics else None,

        "policy_logp_sum_mrr": float(np.mean([m["policy_logp_sum_reciprocal_rank"] for m in metrics])) if metrics else None,

        "policy_logp_sum_top1": float(np.mean([m["policy_logp_sum_top1"] for m in metrics])) if metrics else None,

        "policy_logp_sum_slate_probability": float(np.mean([m["policy_logp_sum_slate_probability"] for m in metrics])) if metrics else None,

        "dpo_reward_pairwise_accuracy": float(np.mean([m["dpo_reward_pairwise_accuracy"] for m in metrics])) if metrics else None,

        "dpo_reward_mean_margin": float(np.mean([m["dpo_reward_mean_margin"] for m in metrics])) if metrics else None,

        "dpo_reward_slate_probability": float(np.mean([m["dpo_reward_slate_probability"] for m in metrics])) if metrics else None,

    }

    return summary, metrics





def evaluate(rows):

    fixed_summary, fixed_metrics = evaluate_fixed_slate(rows)

    generated_summary, generated_metrics = evaluate_generated_bioasq(

        policy_model,

        tokenizer,

        generated_validation_rows,

        max_seq_length=MAX_SEQ_LENGTH,

        max_new_tokens=GENERATED_EVAL_MAX_NEW_TOKENS,

        summary_reference_mode=GENERATED_EVAL_SUMMARY_REFERENCE_MODE,

        do_sample=GENERATED_EVAL_DO_SAMPLE,

        temperature=GENERATED_EVAL_TEMPERATURE,

        top_p=GENERATED_EVAL_TOP_P,

        empty_cuda_cache_per_generation=GENERATED_EVAL_EMPTY_CUDA_CACHE,

        progress_factory=tqdm,

        progress_desc="Generated BioASQ validation",
        official_output_dir=OUTPUT_DIR / 'official_generated_eval' / f'step_{optimizer_step}',
        official_model_label=RUN_NAME,

    )

    fixed_by_qid = {row["question_id"]: row for row in fixed_metrics}

    generated_by_qid = {row["question_id"]: row for row in generated_metrics}

    merged_metrics = []

    for question_id in sorted(set(fixed_by_qid) | set(generated_by_qid)):

        fixed_row = fixed_by_qid.get(question_id, {})

        generated_row = generated_by_qid.get(question_id, {})

        merged_metrics.append({

            "question_id": question_id,

            "loss": fixed_row.get("loss"),

            "fixed_slate_policy_logp_sum_chosen_rank": fixed_row.get("policy_logp_sum_chosen_rank"),

            "fixed_slate_dpo_reward_pairwise_accuracy": fixed_row.get("dpo_reward_pairwise_accuracy"),
            "fixed_slate_dpo_reward_mean_margin": fixed_row.get("dpo_reward_mean_margin"),

            "prediction": generated_row.get("prediction"),

            "gold_output": generated_row.get("gold_output"),

            "generated_mrr": generated_row.get("mrr"),

            "generated_strict_accuracy": generated_row.get("strict_accuracy"),

            "generated_lenient_accuracy": generated_row.get("lenient_accuracy"),

            "generated_primary_score": generated_row.get("primary_score"),

            "generated_prediction_count": generated_row.get("prediction_count"),

            "generated_gold_count": generated_row.get("gold_count"),

        })

    summary = {

        "question_count": fixed_summary.get("question_count"),

        "val_loss": fixed_summary.get("val_loss"),

        "mrr": generated_summary.get("mrr"),

        "top1": generated_summary.get("strict_accuracy"),

        "lenient_accuracy": generated_summary.get("lenient_accuracy"),

        "generated_question_count": generated_summary.get("question_count"),
        "scoring_backend": generated_summary.get("scoring_backend"),

        "fixed_slate_policy_logp_sum_mrr": fixed_summary.get("policy_logp_sum_mrr"),

        "fixed_slate_dpo_reward_pairwise_accuracy": fixed_summary.get("dpo_reward_pairwise_accuracy"),
        "fixed_slate_dpo_reward_mean_margin": fixed_summary.get("dpo_reward_mean_margin"),

        "fixed_slate_rank_definition": "policy_logp_sum ranks raw completion log-probability sums; DPO reward diagnostics report tie-aware pairwise accuracy and chosen-minus-negative reward margins",

    }

    return summary, merged_metrics





def save_checkpoint_bundle(target_dir: Path):

    target_dir.mkdir(parents=True, exist_ok=True)

    policy_model.save_pretrained(target_dir)

    tokenizer.save_pretrained(target_dir)





train_updates_per_epoch = (len(pairwise_train_rows) + GRADIENT_ACCUMULATION_STEPS - 1) // GRADIENT_ACCUMULATION_STEPS

total_optimizer_steps = train_updates_per_epoch * NUM_TRAIN_EPOCHS

optimizer = torch.optim.RMSprop(

    [p for p in policy_model.parameters() if p.requires_grad],

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

)

scheduler = get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=WARMUP_STEPS,

    num_training_steps=max(total_optimizer_steps, WARMUP_STEPS + 1),

)

best_val_loss = None

best_mrr = None

best_loss_summary = None

best_mrr_summary = None

best_loss_metrics = None

best_mrr_metrics = None

history = []

eval_history = []

bad_evals_loss = 0

bad_evals_mrr = 0

micro_step = 0

optimizer_step = 0

start_time = time.time()

should_stop = False


# Record the untouched SFT policy before the first DPO update.
initial_eval_summary, initial_eval_metrics = evaluate(validation_cached)
initial_eval_summary.update({"epoch": 0, "step": 0, "evaluation_phase": "initial"})
eval_history.append(dict(initial_eval_summary))
best_val_loss = initial_eval_summary["val_loss"]
best_mrr = initial_eval_summary["mrr"]
best_loss_summary = dict(initial_eval_summary)
best_mrr_summary = dict(initial_eval_summary)
best_loss_metrics = initial_eval_metrics
best_mrr_metrics = initial_eval_metrics
save_checkpoint_bundle(OUTPUT_DIR / "best_model_by_loss")
save_checkpoint_bundle(OUTPUT_DIR / "best_model_by_mrr")
write_json(OUTPUT_DIR / "step_zero_evaluation_summary.json", initial_eval_summary)


for epoch in range(NUM_TRAIN_EPOCHS):

    epoch_rows = list(pairwise_train_rows)

    random.Random(SEED + epoch).shuffle(epoch_rows)

    progress = tqdm(epoch_rows, desc=f"Epoch {epoch + 1}/{NUM_TRAIN_EPOCHS}")

    optimizer.zero_grad(set_to_none=True)

    for row_index, row in enumerate(progress):

        group_start = (row_index // GRADIENT_ACCUMULATION_STEPS) * GRADIENT_ACCUMULATION_STEPS

        group_size = min(GRADIENT_ACCUMULATION_STEPS, len(epoch_rows) - group_start)

        pairs = [(row["prompt"], row["chosen"]), (row["prompt"], row["rejected"])]

        policy_scores = sequence_logps(policy_model, pairs, require_grad=True)

        ref_chosen = torch.tensor(row["reference_chosen_logp_sum"], dtype=torch.float32, device=DEVICE)

        ref_rejected = torch.tensor(row["reference_rejected_logp_sum"], dtype=torch.float32, device=DEVICE)

        loss, chosen_reward, rejected_reward = pairwise_loss(policy_scores[0], policy_scores[1], ref_chosen, ref_rejected)

        (loss / group_size).backward()

        micro_step += 1

        is_accumulation_boundary = (row_index + 1) % GRADIENT_ACCUMULATION_STEPS == 0

        is_epoch_end = row_index == len(epoch_rows) - 1



        if is_accumulation_boundary or is_epoch_end:

            torch.nn.utils.clip_grad_norm_([p for p in policy_model.parameters() if p.requires_grad], 1.0)

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad(set_to_none=True)

            optimizer_step += 1



        history.append({

            "epoch": epoch + 1,

            "micro_step": micro_step,

            "step": optimizer_step,

            "question_id": row["question_id"],

            "negative_index": row["negative_index"],

            "loss": float(loss.detach().cpu()),

            "chosen_reward": float(chosen_reward.detach().cpu()),

            "rejected_reward": float(rejected_reward.detach().cpu()),

            "learning_rate": scheduler.get_last_lr()[0],

        })

        progress.set_postfix(loss=f"{history[-1]['loss']:.4f}", update=optimizer_step)



        del pairs, policy_scores, ref_chosen, ref_rejected, loss, chosen_reward, rejected_reward

        if EMPTY_CACHE_EVERY_STEPS > 0 and micro_step % EMPTY_CACHE_EVERY_STEPS == 0:

            gc.collect()

            torch.cuda.empty_cache()



        if is_accumulation_boundary or is_epoch_end:

            if EVAL_EVERY_STEPS > 0 and optimizer_step % EVAL_EVERY_STEPS == 0:

                summary, metrics = evaluate(validation_cached)

                summary.update({"epoch": epoch + 1, "step": optimizer_step, "evaluation_phase": "scheduled"})

                eval_history.append(summary)



                improved_loss = best_val_loss is None or summary["val_loss"] < best_val_loss

                improved_mrr = best_mrr is None or summary["mrr"] > best_mrr



                if improved_loss:

                    best_val_loss = summary["val_loss"]

                    best_loss_summary = dict(summary)

                    best_loss_metrics = metrics

                    bad_evals_loss = 0

                    save_checkpoint_bundle(OUTPUT_DIR / "best_model_by_loss")

                else:

                    bad_evals_loss += 1



                if improved_mrr:

                    best_mrr = summary["mrr"]

                    best_mrr_summary = dict(summary)

                    best_mrr_metrics = metrics

                    bad_evals_mrr = 0

                    save_checkpoint_bundle(OUTPUT_DIR / "best_model_by_mrr")

                else:

                    bad_evals_mrr += 1



                if (

                    EARLY_STOPPING_PATIENCE > 0

                    and bad_evals_loss >= EARLY_STOPPING_PATIENCE

                    and bad_evals_mrr >= EARLY_STOPPING_PATIENCE

                ):

                    should_stop = True

                    break

                gc.collect()

                torch.cuda.empty_cache()

    if should_stop:

        break



elapsed = time.time() - start_time

print(f"Training finished in {elapsed / 60:.2f} minutes")

# The final adapter may be between scheduled evaluation boundaries.
if not eval_history or eval_history[-1]["step"] != optimizer_step:

    summary, metrics = evaluate(validation_cached)

    summary.update({"epoch": epoch + 1, "step": optimizer_step, "evaluation_phase": "final"})

    eval_history.append(summary)

    if summary["val_loss"] < best_val_loss:

        best_val_loss = summary["val_loss"]

        best_loss_summary = dict(summary)

        best_loss_metrics = metrics

        save_checkpoint_bundle(OUTPUT_DIR / "best_model_by_loss")

    if summary["mrr"] > best_mrr:

        best_mrr = summary["mrr"]

        best_mrr_summary = dict(summary)

        best_mrr_metrics = metrics

        save_checkpoint_bundle(OUTPUT_DIR / "best_model_by_mrr")
else:

    eval_history[-1]["evaluation_phase"] = "final"



policy_model.save_pretrained(OUTPUT_DIR / "final_adapter")

tokenizer.save_pretrained(OUTPUT_DIR / "final_adapter")


In [ ]:
def write_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False, sort_keys=True)


def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True))
            handle.write("\n")

history_df = pd.DataFrame(history)
eval_df = pd.DataFrame(eval_history)
loss_metrics_df = pd.DataFrame(best_loss_metrics)
mrr_metrics_df = pd.DataFrame(best_mrr_metrics)

history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
eval_df.to_csv(OUTPUT_DIR / "evaluation_history.csv", index=False)
loss_metrics_df.to_csv(OUTPUT_DIR / "best_validation_metrics_by_loss.csv", index=False)
mrr_metrics_df.to_csv(OUTPUT_DIR / "best_validation_metrics_by_mrr.csv", index=False)
write_jsonl(OUTPUT_DIR / "train_questions_with_reference.jsonl", train_cached)
write_jsonl(OUTPUT_DIR / "validation_questions_with_reference.jsonl", validation_cached)
write_json(OUTPUT_DIR / "best_validation_summary_by_loss.json", best_loss_summary)
write_json(OUTPUT_DIR / "best_validation_summary_by_mrr.json", best_mrr_summary)

last_microbatch_training_loss = None if history_df.empty else float(history_df["loss"].iloc[-1])
final_training_epoch = None if history_df.empty else int(history_df["epoch"].iloc[-1])
final_epoch_history = history_df[history_df["epoch"] == final_training_epoch] if final_training_epoch is not None else history_df
final_training_epoch_mean_loss = None if final_epoch_history.empty else float(final_epoch_history["loss"].mean())
final_eval_loss = None if eval_df.empty else float(eval_df["val_loss"].iloc[-1])
final_eval_mrr = None if eval_df.empty else float(eval_df["mrr"].iloc[-1])
final_eval_top1 = None if eval_df.empty else float(eval_df["top1"].iloc[-1])
final_evaluation_phase = None if eval_df.empty else str(eval_df["evaluation_phase"].iloc[-1])

summary = {
    "method": "pairwise_standard_dpo",
    "smoke_test": SMOKE_TEST,
    "smoke_tag": SMOKE_TAG if SMOKE_TEST else None,
    "smoke_train_questions": len(train_question_rows),
    "smoke_validation_questions": len(validation_question_rows),
    "model_ref": str(MODEL_REF),
    "base_model_ref": str(BASE_MODEL_REF),
    "train_jsonl": str(TRAIN_JSONL),
    "validation_jsonl": str(VALIDATION_JSONL),
    "validation_gold_eval_input": str(VALIDATION_GOLD_EVAL_INPUT),
    "seed": SEED,
    "beta": BETA,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_TRAIN_EPOCHS,
    "train_question_count": len(train_cached),
    "train_pair_count": len(pairwise_train_rows),
    "validation_question_count": len(validation_cached),
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "stopping_rule": "stop only after neither validation loss nor validation MRR has improved for the configured patience",
    "last_microbatch_training_loss": last_microbatch_training_loss,
    "final_training_epoch": final_training_epoch,
    "final_training_epoch_mean_loss": final_training_epoch_mean_loss,
    "final_eval_loss": final_eval_loss,
    "final_eval_mrr": final_eval_mrr,
    "final_eval_top1": final_eval_top1,
    "final_evaluation_phase": final_evaluation_phase,
    "mrr_definition": "generated answers scored by the official BioASQ Java Phase-B factoid evaluator",
    "best_by_loss": {
        "val_loss": None if best_loss_summary is None else best_loss_summary.get("val_loss"),
        "mrr": None if best_loss_summary is None else best_loss_summary.get("mrr"),
        "top1": None if best_loss_summary is None else best_loss_summary.get("top1"),
        "summary": best_loss_summary,
        "checkpoint_dir": str(OUTPUT_DIR / "best_model_by_loss"),
    },
    "best_by_mrr": {
        "val_loss": None if best_mrr_summary is None else best_mrr_summary.get("val_loss"),
        "mrr": None if best_mrr_summary is None else best_mrr_summary.get("mrr"),
        "top1": None if best_mrr_summary is None else best_mrr_summary.get("top1"),
        "summary": best_mrr_summary,
        "checkpoint_dir": str(OUTPUT_DIR / "best_model_by_mrr"),
    },
}
write_json(OUTPUT_DIR / "summary.json", summary)
write_json(OUTPUT_DIR / "manifest.json", {
    "train_jsonl": str(TRAIN_JSONL),
    "validation_jsonl": str(VALIDATION_JSONL),
    "dataset_variant": DATASET_VARIANT,
    "dataset_label": DATASET_LABEL,
    "allow_unreviewed_span_candidates": ALLOW_UNREVIEWED_SPAN_CANDIDATES,
    "allow_unreviewed_candidate_bank_span_candidates": ALLOW_UNREVIEWED_CANDIDATE_BANK_SPAN_CANDIDATES,
    "model_ref": str(MODEL_REF),
    "model_variant": MODEL_VARIANT,
    "prompt_id": DPO_PROMPT_ID,
    "prompt_variant": PROMPT_VARIANT,
    "prompt_registry_path": str(PROMPT_REGISTRY_PATH),
    "base_model_ref": str(BASE_MODEL_REF),
    "output_dir": str(OUTPUT_DIR),
    "method": "pairwise_standard_dpo",
    "beta": BETA,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_TRAIN_EPOCHS,
    "summary_json": str(OUTPUT_DIR / "summary.json"),
    "best_model_by_loss": str(OUTPUT_DIR / "best_model_by_loss"),
    "best_model_by_mrr": str(OUTPUT_DIR / "best_model_by_mrr"),
})

plt.figure(figsize=(10, 4))
ax1 = plt.gca()
if not history_df.empty:
    train_plot_df = history_df.groupby("step", as_index=False)["loss"].mean()
    ax1.plot(train_plot_df["step"], train_plot_df["loss"], label="train loss", alpha=0.7)
if not eval_df.empty:
    ax1.plot(eval_df["step"], eval_df["val_loss"], label="val loss", marker="o")
ax2 = ax1.twinx()
if not eval_df.empty:
    ax2.plot(eval_df["step"], eval_df["mrr"], label="val MRR", color="tab:green", marker="s")
plt.title(f"Pairwise DPO Training Curves ({SMOKE_TAG if SMOKE_TEST else 'full'})")
ax1.set_xlabel("optimizer step")
ax1.set_ylabel("loss")
ax2.set_ylabel("MRR")
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="upper right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=200)
plt.show()

summary
